# Training models taking account the Data Reducing

## Loading the Database

In [2]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import numpy as np

IMG_SIZE = 512
BATCH_SIZE = 8
NUM_CLASSES = 1
CROP = "wheat"  # "wheat" or "sorghum"

dataset_path = f"../data/{CROP}/"

class SegmentationDataset(Dataset):
    def __init__(self, root_dir, train=True, num_augs_to_use=1, seed=None):
        self.root_dir = root_dir
        self.train = train
        self.img_dir = os.path.join(root_dir, "train/images" if train else "test/images")
        self.mask_dir = os.path.join(root_dir, "train/masks" if train else "test/masks")
        self.num_augs_to_use = num_augs_to_use
        self.rng = random.Random(seed)

        # Find base filenames (without augmentation suffix)
        base_files = set()
        for f in os.listdir(self.img_dir):
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                base = f.split('_aug')[0] if '_aug' in f else os.path.splitext(f)[0]
                base_files.add(base)

        # For each base file, collect available augmentations
        self.samples = []
        for base in sorted(base_files):
            aug_indices = []
            for aug in range(1, 6):
                img_name = f"{base}_aug{aug}.png"
                mask_name = f"{base}_aug{aug}.png"
                if os.path.exists(os.path.join(self.img_dir, img_name)) and os.path.exists(os.path.join(self.mask_dir, mask_name)):
                    aug_indices.append(aug)
            # If original (no _aug) exists, add as aug=0
            img_name = f"{base}.png"
            mask_name = f"{base}.png"
            if os.path.exists(os.path.join(self.img_dir, img_name)) and os.path.exists(os.path.join(self.mask_dir, mask_name)):
                aug_indices.append(0)
            # Randomly select num_augs_to_use augmentations for this base
            if len(aug_indices) > 0:
                selected_augs = self.rng.sample(aug_indices, min(num_augs_to_use, len(aug_indices)))
                for aug in selected_augs:
                    self.samples.append((base, aug))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        base, aug = self.samples[idx]
        if aug == 0:
            img_name = f"{base}.png"
            mask_name = f"{base}.png"
        else:
            img_name = f"{base}_aug{aug}.png"
            mask_name = f"{base}_aug{aug}.png"

        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # grayscale mask

        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        mask = torch.from_numpy(np.array(mask)).float().unsqueeze(0)
        mask = (mask > 0).float()

        return image, mask

# Example usage:
train_dataset = SegmentationDataset(dataset_path, train=True, num_augs_to_use=4, seed=666)
test_dataset = SegmentationDataset(dataset_path, train=False, num_augs_to_use=4, seed=666)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"✅ Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

ValueError: num_samples should be a positive integer value, but got num_samples=0